# 06 · Contraction with einsum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Contracción con einsum** — Aprender una sola regla de índices, comprobarla con datos reales y cambiar las entradas de forma interactiva para identificar qué índice desaparece.

Use one index rule on real data, then change the inputs interactively to test which index disappears.

## What you will be able to do

- Explain the `einsum` rule: indices missing after `->` are summed over; indices that remain are kept.
- Contract the colour axis of a real microscopy image and explain what the RGB sliders change — and what they do not change.
- Read trace, transpose and matrix multiplication as index operations on pixel patches cut from real handwritten digits.
- Build all 3,229,209 pairwise similarities between 1,797 real digit images and explore retrieval with cosine similarity versus raw dot product.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.datasets import load_digits
from skimage import data

# Enable ipywidgets in Google Colab when available.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Real colour images.
photo = data.immunohistochemistry().astype(float)          # (512, 512, 3)
batch = np.stack([photo, data.astronaut().astype(float)]) # (2, 512, 512, 3)
w = np.array([0.2125, 0.7154, 0.0721])                    # RGB -> grayscale weights

# Real handwritten digits (UCI Optical Recognition dataset, packaged by sklearn).
digits = load_digits()
digit_images = digits.images.astype(float)                 # (1797, 8, 8)

# Small 2x2 matrices for Exercise 2 are NOT invented numbers:
# they are central pixel patches from two real digit images.
A = digit_images[0, 2:4, 2:4]
B = digit_images[1, 2:4, 2:4]

print("photo:", photo.shape, "batch:", batch.shape)
print("digits:", digit_images.shape, "labels:", digits.target.shape)
print(
    "Exercise 2 patches come from digit labels:",
    digits.target[0],
    "and",
    digits.target[1],
)
print("A =\n", A)
print("B =\n", B)

## Why this matters

A contraction is not just a matrix-algebra trick. The same operation appears when an image model removes a colour axis, when linear algebra multiplies matrices, and when a retrieval system compares one query vector with thousands or millions of candidates.

> 🇪🇸 Una contracción no es solo un truco de álgebra matricial. La misma operación aparece cuando un modelo elimina el eje de color de una imagen, cuando el álgebra lineal multiplica matrices y cuando un sistema de búsqueda compara un vector consulta con miles o millones de candidatos.

### One rule for the whole notebook

If an index appears in the inputs but **not** after `->`, it is **summed over**. If it appears after `->`, it is **kept**.

Examples:

- `hwc,c->hw`: `c` disappears → contract colour.
- `ik,kj->ij`: `k` disappears → matrix multiplication.
- `id,jd->ij`: `d` disappears → every pair of row vectors gets one similarity score.

### Predict → Run → Explain

Before each exercise, predict **which index disappears and what the output shape must be**. After running the code, explain what information the contraction kept.

> 🇪🇸 **Predice → Ejecuta → Explica:** antes de cada ejercicio, predice **qué índice desaparece y cuál debe ser la forma de salida**. Después de ejecutar, explica qué información conservó la contracción.

## Exercise 1 — contract the colour axis of a real image

### What are you looking at?

`photo` is a **real microscopy image** from `skimage.data.immunohistochemistry()` with shape `(512, 512, 3)`:

- `h = 512`: image rows / height
- `w = 512`: image columns / width
- `c = 3`: red, green and blue channels

The vector `w = [0.2125, 0.7154, 0.0721]` is **not another dataset**. It is a transformation rule: three weights telling us how much each colour channel contributes to the grayscale result.

### What is the mathematical idea?

In:

`hwc,c->hw`

the index `c` appears in the inputs but **disappears after `->`**, so `einsum` multiplies each colour channel by its weight and **sums over colour**. The `h` and `w` indices survive, so the output remains an image of shape `(512, 512)`.

### What should you try?

1. Predict the output shape before running anything.
2. Solve the TODO with one `einsum`.
3. Open the folded solution and move the **R/G/B sliders**.
4. Notice that the picture changes, but the index rule `hwc,c->hw` does not.

> 🇪🇸 **¿Qué estás viendo?** `photo` es una imagen real de microscopía de forma `(512, 512, 3)`: alto, ancho y tres canales RGB. El vector `w` no es un conjunto de datos; es una regla de transformación. En `hwc,c->hw`, el índice `c` desaparece después de `->`, por eso se multiplica cada canal por su peso y luego se suma sobre color. **Prueba:** predice la forma, resuelve el `einsum` y después mueve los sliders R/G/B. La imagen cambia, pero la regla de índices permanece igual.


In [ ]:
# TODO 1: Use einsum to convert `photo` to grayscale by contracting the colour
#         axis against w. Expected result: (512, 512).
#
# TODO 2: Do the same for the whole real-image batch in ONE einsum call.
#         Expected result: (2, 512, 512).
#
# Explain in one sentence:
# - which index is contracted?
# - which indices survive?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
gray       = np.einsum('hwc,c->hw',   photo, w)       # (512, 512)
gray_batch = np.einsum('nhwc,c->nhw', batch, w)       # (2, 512, 512)

print("single image:", photo.shape, "->", gray.shape)
print("batch:", batch.shape, "->", gray_batch.shape)
print("contracted index: c | kept indices: h,w (and n for the batch)")

# Static before/after visual: real photograph in, computed contraction out.
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(photo / 255)
axes[0].set_title("real input — axes h, w, c")
axes[1].imshow(gray, cmap="gray")
axes[1].set_title("'hwc,c->hw' — c is gone")
for ax in axes:
    ax.axis("off")
fig.suptitle("c is missing after ->, so colour is SUMMED OVER; h and w are KEPT")
plt.tight_layout()
plt.show()

# Interactive lab: change the contraction weights while keeping the SAME real image
# and the SAME index rule. continuous_update=False keeps Colab responsive.
r_slider = widgets.FloatSlider(
    value=float(w[0]), min=0.0, max=1.0, step=0.025,
    description="R", readout_format=".3f", continuous_update=False
)
g_slider = widgets.FloatSlider(
    value=float(w[1]), min=0.0, max=1.0, step=0.025,
    description="G", readout_format=".3f", continuous_update=False
)
b_slider = widgets.FloatSlider(
    value=float(w[2]), min=0.0, max=1.0, step=0.025,
    description="B", readout_format=".3f", continuous_update=False
)

def explore_colour_contraction(r, g, b):
    weights = np.array([r, g, b], dtype=float)
    live_gray = np.einsum('hwc,c->hw', photo, weights)

    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.3))
    axes[0].imshow(photo / 255)
    axes[0].set_title("same real input")
    axes[1].imshow(live_gray, cmap="gray")
    axes[1].set_title(f"R={r:.3f}, G={g:.3f}, B={b:.3f}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("EN: c disappears -> colour is contracted; h and w survive.")
    print("ES: c desaparece -> el color se contrae; h y w permanecen.")
    print(f"einsum: hwc,c->hw | weight sum / suma de pesos = {weights.sum():.3f}")

rgb_output = widgets.interactive_output(
    explore_colour_contraction,
    {"r": r_slider, "g": g_slider, "b": b_slider},
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Interactive lab:</b> move R/G/B. "
            "The pixels stay real; only the contraction weights change."
        ),
        widgets.HBox([r_slider, g_slider, b_slider]),
        rgb_output,
    ])
)


## Exercise 2 — Chapter 2 operations on real digit pixels

### Where do matrices `A` and `B` come from?

They are **not hand-written toy numbers**. Each `2×2` matrix is the central pixel patch of one real `8×8` handwritten digit from `sklearn.datasets.load_digits()`.

Pixel intensities in this dataset run from **0 to 16**. The matrices are intentionally tiny so you can check the arithmetic by hand while still operating on observed data.

### What are the three operations teaching?

- **Trace — `ii->`**: the repeated `i` disappears, so the diagonal is summed to one scalar.
- **Transpose — `ij->ji`**: no index disappears; the axes are only reordered.
- **Matrix product — `ik,kj->ij`**: the shared `k` disappears, so `k` is the contracted dimension; `i` and `j` survive.

### What should you try?

Use the **operation selector** after solving the TODO. Switch among trace, transpose and matrix product and say aloud:

> “Which index disappeared? Which indices survived?”

That sentence is more important than memorising the strings.

> 🇪🇸 **¿De dónde salen `A` y `B`?** No son números inventados: cada matriz `2×2` es un recorte central de píxeles de un dígito manuscrito real `8×8`. Las intensidades van de **0 a 16**. **Traza:** `ii->` elimina `i` y suma la diagonal. **Transpuesta:** `ij->ji` no elimina índices, solo cambia su orden. **Producto matricial:** `ik,kj->ij` elimina `k`, por lo que `k` es la dimensión contraída. Usa el selector y pregúntate siempre: **¿qué índice desapareció y cuáles sobrevivieron?**


In [ ]:
# TODO 3: Rewrite these Chapter 2 operations as einsum and check each against
#         NumPy:
#
#   (a) trace of A           -> scalar
#   (b) transpose of A       -> (2, 2)
#   (c) matrix product A @ B -> (2, 2)
#
# For each expression, say which index is:
# - summed over,
# - only relabelled/reordered,
# - or kept.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
trace_e = np.einsum('ii->', A)
transpose_e = np.einsum('ij->ji', A)
product_e = np.einsum('ik,kj->ij', A, B)

print("A came from real digit label", digits.target[0])
print(A)
print("B came from real digit label", digits.target[1])
print(B)
print("\ntrace:", trace_e, "| NumPy:", np.trace(A))
print("transpose:\n", transpose_e)
print("matrix product:\n", product_e)

assert np.allclose(trace_e, np.trace(A))
assert np.allclose(transpose_e, A.T)
assert np.allclose(product_e, A @ B)
print("\nall three einsum results agree with NumPy")

# Context: show the real 8x8 source digits and highlight the 2x2 patches.
fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))
for ax, idx, name in zip(axes, [0, 1], ["A", "B"]):
    ax.imshow(digit_images[idx], cmap="gray_r", interpolation="nearest", vmin=0, vmax=16)
    ax.add_patch(
        plt.Rectangle((1.5, 1.5), 2, 2, fill=False, linewidth=2)
    )
    ax.set_title(f"{name}: real digit label {digits.target[idx]}\nred box = 2×2 patch")
    ax.axis("off")
plt.tight_layout()
plt.show()

operation = widgets.ToggleButtons(
    options=[
        ("Trace / Traza", "trace"),
        ("Transpose / Transpuesta", "transpose"),
        ("Matrix product / Producto", "product"),
    ],
    value="product",
    description="",
)

def explain_operation(op):
    if op == "trace":
        expr = "ii->"
        result = np.einsum("ii->", A)
        en = "i is repeated and disappears -> sum the diagonal."
        es = "i se repite y desaparece -> se suma la diagonal."
        print("A =\n", A)
        print(f"einsum('{expr}', A) =", result)
    elif op == "transpose":
        expr = "ij->ji"
        result = np.einsum("ij->ji", A)
        en = "No index disappears -> only reorder the axes."
        es = "Ningún índice desaparece -> solo se reordenan los ejes."
        print("A =\n", A)
        print(f"einsum('{expr}', A) =\n", result)
    else:
        expr = "ik,kj->ij"
        result = np.einsum("ik,kj->ij", A, B)
        en = "k is shared and disappears -> contract k; keep i and j."
        es = "k es compartido y desaparece -> contrae k; conserva i y j."
        print("A =\n", A)
        print("B =\n", B)
        print(f"einsum('{expr}', A, B) =\n", result)

    print("EN:", en)
    print("ES:", es)

operation_output = widgets.interactive_output(
    explain_operation,
    {"op": operation},
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Interactive index explorer / Explorador interactivo de índices:</b> "
            "choose an operation and identify the disappearing index. / "
            "elige una operación e identifica el índice que desaparece."
        ),
        operation,
        operation_output,
    ])
)


## Exercise 3 — 3,229,209 similarities from 1,797 real digit images

### Why do these digit images look pixelated?

This is **intentional and important**: the original `sklearn` digits dataset stores each handwritten digit at only **8×8 pixels**.

So each image has exactly:

`8 × 8 = 64 real measured pixel features`

The blocky appearance is **not a bad download, compression error or broken image**. It is the original resolution of the dataset. We display it with `interpolation="nearest"` so the notebook does **not invent smooth pixels that were never measured**.

That low resolution is actually useful here: after flattening, every digit becomes a 64-dimensional vector, so the contraction is easy to connect directly to the pixels.

### What does `id,jd->ij` mean?

- `i`: query-image index — kept
- `j`: candidate-image index — kept
- `d`: 64 pixel features — **disappears**, so it is contracted

The result therefore has shape `(1797, 1797)`: one similarity score for every pair of real digit images.

### Raw dot product vs cosine similarity

Both use the **same `einsum` contraction**. The difference is what happens before it:

- **Raw dot product** also rewards vector magnitude — roughly, how much total “ink” or intensity an image has.
- **Cosine similarity** first normalizes each 64-pixel vector to unit length, so the comparison focuses more on the **pattern/direction** of the pixels.

For query image `14` (true label `4`), the strongest raw-dot match is a `1`, while the strongest cosine matches are `4`s. That is a real example of why preprocessing changes the meaning of “similar”.

### What should you try?

Use the retrieval explorer:

1. Move **Query / Consulta** to choose any of the 1,797 real digits.
2. Switch **Cosine / Coseno** ↔ **Raw dot / P. punto**.
3. Change **Top k / Vecinos**.
4. Compare how many retrieved images have the same label as the query.

> 🇪🇸 **¿Por qué se ven pixelados los dígitos?** Porque el dataset original guarda cada dígito con solo **8×8 píxeles**. No es mala calidad de descarga ni un error: son exactamente **64 mediciones reales** por imagen. Se muestran con interpolación `nearest` para no inventar píxeles suaves que nunca fueron observados. En `id,jd->ij`, `d` representa esas 64 características y desaparece; `i` y `j` permanecen, por eso obtenemos una matriz `(1797,1797)` con una similitud para cada par. El producto punto crudo también depende de la magnitud/intensidad; el coseno normaliza primero y compara más la forma del patrón. Usa el explorador para cambiar consulta, métrica y número de vecinos.


In [ ]:
# TODO 4: Reshape all real digit images to D with shape (1797, 64).
#
# TODO 5: Compute every raw dot-product similarity with ONE einsum:
#         'id,jd->ij' -> expected shape (1797, 1797).
#
# TODO 6: Normalize every row of D to unit length, then repeat the SAME einsum
#         to obtain cosine similarity C.
#
# TODO 7: Use query_idx = 14 (true label 4).
#         - exclude the query from matching itself,
#         - find the best raw-dot-product match,
#         - find the five best cosine-similarity matches,
#         - compare their labels with digits.target[query_idx].
#
# Predict first: which index disappears in 'id,jd->ij'?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
D = digit_images.reshape(len(digit_images), -1)       # (1797, 64)

S = np.einsum('id,jd->ij', D, D)                     # (1797, 1797)
print("raw similarity matrix:", S.shape, "=", S.size, "pairwise scores")

norms = np.linalg.norm(D, axis=1, keepdims=True)
Dn = D / np.where(norms == 0, 1.0, norms)
C = np.einsum('id,jd->ij', Dn, Dn)

assert S.shape == (1797, 1797)
assert C.shape == (1797, 1797)
assert np.allclose(C.diagonal(), 1.0)

query_idx = 14
query_label = digits.target[query_idx]

raw_scores = S[query_idx].copy()
cos_scores = C[query_idx].copy()
raw_scores[query_idx] = -np.inf
cos_scores[query_idx] = -np.inf

raw_top1 = int(np.argmax(raw_scores))
cos_top5 = np.argsort(cos_scores)[-5:][::-1]

print("query index / label:", query_idx, "/", query_label)
print("best RAW dot-product match:", raw_top1,
      "label", digits.target[raw_top1],
      "score", round(float(raw_scores[raw_top1]), 3))
print("top 5 COSINE matches:", cos_top5.tolist())
print("top 5 COSINE labels:", digits.target[cos_top5].tolist())
print("top 5 COSINE scores:", np.round(cos_scores[cos_top5], 3).tolist())

# Static retrieval example. nearest preserves the original 8x8 measurements.
fig, axes = plt.subplots(1, 6, figsize=(11, 2.5))
axes[0].imshow(
    digit_images[query_idx], cmap="gray_r",
    interpolation="nearest", vmin=0, vmax=16
)
axes[0].set_title(f"query / consulta\nlabel {query_label}")

for ax, idx in zip(axes[1:], cos_top5):
    ax.imshow(
        digit_images[idx], cmap="gray_r",
        interpolation="nearest", vmin=0, vmax=16
    )
    ax.set_title(f"label {digits.target[idx]}\ncos={C[query_idx, idx]:.3f}")

for ax in axes:
    ax.axis("off")

fig.suptitle(
    "Real 8×8 digits: one contraction -> all-pairs similarity -> retrieval"
)
plt.tight_layout()
plt.show()

query_slider = widgets.IntSlider(
    value=14, min=0, max=len(digit_images) - 1, step=1,
    description="Query:",
    continuous_update=False,
    style={"description_width": "55px"},
)
similarity_toggle = widgets.ToggleButtons(
    options=[("Cosine / Coseno", "cosine"), ("Raw dot / P. punto", "raw")],
    value="cosine",
    description="",
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=8, step=1,
    description="Top k:",
    continuous_update=False,
    style={"description_width": "45px"},
)

def explore_retrieval(query, metric, k):
    matrix = C if metric == "cosine" else S
    scores = matrix[query].copy()
    scores[query] = -np.inf

    top = np.argsort(scores)[-k:][::-1]
    q_label = int(digits.target[query])

    fig, axes = plt.subplots(1, k + 1, figsize=(2.05 * (k + 1), 2.75))
    axes = np.atleast_1d(axes)

    axes[0].imshow(
        digit_images[query], cmap="gray_r",
        interpolation="nearest", vmin=0, vmax=16
    )
    axes[0].set_title(f"query {query}\nlabel {q_label}")
    axes[0].axis("off")

    for ax, idx in zip(axes[1:], top):
        ax.imshow(
            digit_images[idx], cmap="gray_r",
            interpolation="nearest", vmin=0, vmax=16
        )
        score_name = "cos" if metric == "cosine" else "dot"
        ax.set_title(
            f"idx {idx}\nlabel {digits.target[idx]}\n{score_name}={scores[idx]:.3f}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    labels = digits.target[top].astype(int).tolist()
    same_label = sum(label == q_label for label in labels)

    if metric == "cosine":
        en = "Cosine normalizes magnitude first, so the comparison emphasizes pixel-pattern direction."
        es = "El coseno normaliza la magnitud primero, por lo que enfatiza la dirección del patrón de píxeles."
    else:
        en = "Raw dot product also rewards magnitude/intensity, so visually different labels can rank highly."
        es = "El producto punto crudo también premia magnitud/intensidad, por eso pueden aparecer etiquetas distintas."

    print(
        f"metric={metric} | query label={q_label} | "
        f"top-{k} labels={labels} | same-label matches={same_label}/{k}"
    )
    print("EN:", en)
    print("ES:", es)
    print("Index rule / Regla: id,jd->ij | d disappears / desaparece; i and j survive / permanecen.")
    print("Display note / Nota visual: the data are truly 8×8; the pixelated look is the original resolution.")

retrieval_output = widgets.interactive_output(
    explore_retrieval,
    {
        "query": query_slider,
        "metric": similarity_toggle,
        "k": k_slider,
    },
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Interactive retrieval explorer / Explorador interactivo:</b> "
            "choose a real query digit, change the metric, and inspect its neighbours. / "
            "elige un dígito real, cambia la métrica y observa sus vecinos."
        ),
        widgets.HTML(
            "<i>Image quality note / Nota de calidad:</i> these are original 8×8 measurements; "
            "the blocky pixels are the data, not an error. / Son mediciones originales 8×8; "
            "los bloques son los datos, no un error."
        ),
        widgets.HBox([query_slider, similarity_toggle, k_slider]),
        retrieval_output,
    ])
)


## What just happened

You used **one index rule** three times, but each exercise gave the rule a different meaning:

1. **Real microscopy image — `hwc,c->hw`**  
   `c` disappeared, so three colour measurements became one grayscale value at every pixel. The sliders changed the weights, not the rule.

2. **Real digit-pixel matrices — `ik,kj->ij`**  
   `k` disappeared, so the shared dimension was multiplied and summed. Trace and transpose showed that `einsum` can also sum repeated indices or simply reorder them.

3. **1,797 real handwritten digits — `id,jd->ij`**  
   `d` disappeared, so 64 real pixel measurements became one similarity score for every pair of images: **3,229,209 scores**. The interactive explorer showed that preprocessing changes what “similar” means.

### The sentence to remember

> **If an index disappears after `->`, it is summed over. If it remains, it survives in the output.**

The `8×8` digits are deliberately pixelated because **each visible square is one of the 64 measured features**. Keeping that limitation visible makes the tensor operation easier to understand.

> 🇪🇸 Usaste **una sola regla de índices** tres veces: `c` desapareció al convertir color a gris; `k` desapareció en el producto matricial; y `d` desapareció al convertir 64 píxeles en una similitud entre dos dígitos. La frase para recordar es: **si un índice desaparece después de `->`, se suma; si permanece, sobrevive en la salida.** Los dígitos `8×8` se ven pixelados a propósito: cada cuadrado visible es una de las 64 características realmente medidas.

Keep this rule for section 10: `'ijk,ia,jb,kc->abc'` looks longer, but the logic is exactly the same.


---

## Done with this section

Next up: **07 · Inverses and the pseudoinverse** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/07-inverses-and-pseudoinverse.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)